In [2]:
#reading data labels and bboxes
import json
import pandas as pd
import os
import numpy as np

folder = r"Hagrid\ann_train_val"
json_files = os.listdir(folder)
labels = ['one','two_up','two_up_inverted','three','three2','four','palm']

train_path = r"Hagrid\ann_train_val"
picture_path =  r"Hagrid\hagrid_30k"
row = []
missing = 0
for file in labels:
    path = os.path.join(train_path, file + ".json")
    with open(path, "r") as f:
        data = json.load(f)

    for id, sample in data.items():
        img_path = os.path.join(picture_path, f"train_val_{file}", f"{id}.jpg")
        
        if not os.path.exists(img_path):
            missing += 1
            continue
        
        row.append({
            "label": sample['labels'],
            "bbox": sample['bboxes'],
            "path": img_path
        })

print(f"Data: {len(row)}, missing: {missing}")
df = pd.DataFrame(row)
df.to_csv(r'yolo_dataset\df.csv', index=False)

Data: 12461, missing: 186908


In [3]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision.io import read_image

label_map = {
    'one': 0,
    'two_up': 1,
    'two_up_inverted': 2,
    'three': 3,
    'three2': 4,
    'four': 5,
    'palm': 6
}

class Box_Dataset(Dataset):
    def __init__(self, dict, labels):
        self.dict = dict
        
    def __len__(self):
        return (len(self.data))
    
    def __getitem__(self, idx):
        sample = self.dict[idx]
        img = Image.open(sample['path']).convert('RGB')
        w_img, h_img = img.size
        
        bboxes = []
        for box in sample['bbox']:
            x, y, w, h = box
            x1 = x * w_img
            y1 = y * h_img
            x2 = (x + w) * w_img
            y2 = (y + h) * h_img
            bboxes.append([x1, y1, x2, y2])
        labels = torch.tensor([label_map[l] for l in sample['label']],dtype=torch.long)
        
        return img, bboxes, labels

In [4]:
#creatinf yaml file
import os
import shutil
from sklearn.model_selection import train_test_split

OUTPUT_DIR = r"yolo_dataset"
TRAIN_RATIO = 0.8

LABEL_MAP = {
    'one': 0,
    'two_up': 1,
    'two_up_inverted': 2,
    'three': 3,
    'three2': 4,
    'four': 5,
    'palm': 6
}

for split in ['train', 'val']:
    os.makedirs(os.path.join(OUTPUT_DIR, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, 'labels', split), exist_ok=True)


train_data, val_data = train_test_split(row, test_size= 1 - TRAIN_RATIO, random_state=42)
print(f"Train: {len(train_data)}, Val: {len(val_data)}")

def process_split(data, split_name):
    skipped = 0
    for sample in data:
        img_path = sample['path']
        bboxes = sample['bbox']
        labels = sample['label']

        img_name = os.path.basename(img_path)
        stem = os.path.splitext(img_name)[0]

        dst_img = os.path.join(OUTPUT_DIR, 'images', split_name, img_name)
        dst_lbl = os.path.join(OUTPUT_DIR, 'labels', split_name, stem + '.txt')

        shutil.copy2(img_path, dst_img)

        lines = []
        for box, lbl in zip(bboxes, labels):
            if lbl not in LABEL_MAP:
                skipped += 1
                continue
            x, y, w, h = box
            cx = x + w / 2
            cy = y + h / 2
            class_id = LABEL_MAP[lbl]
            lines.append(f"{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")

        with open(dst_lbl, 'w') as f:
            f.write('\n'.join(lines))

    print(f"[{split_name}] zapisano: {len(data) - skipped}, pominięto: {skipped}")

process_split(train_data, 'train')
process_split(val_data, 'val')


yaml_content = f"""path: {OUTPUT_DIR}
train: images/train
val: images/val

nc: {len(LABEL_MAP)}
names: {list(LABEL_MAP.keys())}
"""

yaml_path = os.path.join(OUTPUT_DIR, 'dataset.yaml')
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f"\nGotowe! dataset.yaml zapisany w: {yaml_path}")

Train: 9968, Val: 2493
[train] zapisano: 7691, pominięto: 2277
[val] zapisano: 1934, pominięto: 559

Gotowe! dataset.yaml zapisany w: C:\Users\qmiko\Desktop\Studia\python_projekty\Real-time recognition\yolo_dataset\dataset.yaml
